# Week 56, Day 5: Decoding categories from population activity

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w056/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Accuracy in every time bin

Complete shrink_lda_fit(X, y, gamma): class means, pooled covariance, shrinkage towards the identity, and per-class linear scores w_k = S^-1 mu_k and b_k = -w_k · mu_k / 2. Then compute five-fold cross-validated accuracy in every time bin.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(565)
N, C, n_trials = 100, 4, 240
times = np.arange(-50, 300, 25)                       # bin starts (ms)
patterns = r.normal(size=(C, N)) * 0.2                # category-specific modulation
def time_course(t):
    return np.clip((t - 80) / 50, 0, 1)                # responses begin about 80 ms after onset
cats = r.integers(0, C, n_trials)
shared = r.normal(size=(n_trials, len(times), 1)) * 0.3
log_rate = np.log(2.0) + time_course(times)[None, :, None] * patterns[cats][:, None, :] + shared
counts = r.poisson(np.exp(log_rate)).astype(float)    # trials x bins x neurons

def shrink_lda_fit(X, y, gamma):
    classes = np.arange(C)
    mus = np.array([X[y == k].mean(0) for k in classes])
    Xc = X - mus[y]
    S = Xc.T @ Xc / (len(y) - C)
    S = ...
    W = np.linalg.solve(S, mus.T)                     # D x C
    b = ...
    return W, b

def cv_acc(X, y, gamma=0.2, K=5):
    folds = np.array_split(np.arange(len(y)), K); correct = 0
    for te in folds:
        tr = np.setdiff1d(np.arange(len(y)), te)
        W, b = shrink_lda_fit(X[tr], y[tr], gamma)
        correct += np.sum((X[te] @ W + b).argmax(1) == y[te])
    return correct / len(y)

acc_t = np.array([cv_acc(counts[:, j, :], cats) for j in range(len(times))])
for t_, a in zip(times, acc_t):
    if t_ % 50 == 0: print(f"{t_:4d} ms: accuracy {a:.2f}")
plt.plot(times + 12.5, acc_t, "o-"); plt.axhline(0.25, color="grey", ls="--"); plt.axvline(0, color="k")
plt.xlabel("time from image onset (ms)"); plt.ylabel("cross-validated accuracy"); plt.show()

In [ ]:
import numpy as np
def test_lda():
    "shrink_lda_fit returns one weight vector and bias per class"
    W, b = shrink_lda_fit(counts[:, 5, :], cats, 0.2)
    assert W.shape == (N, 4) and b.shape == (4,)
def test_baseline():
    "accuracy before stimulus onset is near chance (0.25)"
    assert np.all(np.abs(acc_t[times < 50] - 0.25) < 0.08)
def test_peak():
    "accuracy after 150 ms exceeds 0.6"
    assert acc_t[times >= 150].max() > 0.6

test_lda()
test_baseline()
test_peak()
print("All checks passed")

### Exercise: Accuracy against population size

Sum counts over the bins from 200 to 275 ms. For each population size, average the cross-validated accuracy over 10 random subsets of neurons.

In [ ]:
late = counts[:, times >= 200, :].sum(1)
sizes = [2, 5, 10, 25, 50, 100]
acc_n = np.zeros(len(sizes))
for i, m in enumerate(sizes):
    accs = []
    for rep in range(10):
        sub = r.choice(N, m, replace=False)
        accs.append(...)
    acc_n[i] = np.mean(accs)
    print(f"{m:3d} neurons: accuracy {acc_n[i]:.2f}")

In [ ]:
import numpy as np
def test_grows():
    "accuracy grows with the number of neurons"
    assert acc_n[-1] > acc_n[0] + 0.2
def test_avg():
    "each entry averages 10 random subsets"
    assert acc_n.shape == (len(sizes),)

test_grows()
test_avg()
print("All checks passed")

### Exercise: Temporal generalisation

Simulate a dynamic code: from 80 to 175 ms the categories use one set of patterns, from 175 ms another. Then fill G[i, j] with the accuracy of a decoder trained on bin i (all trials in the training folds) and tested on bin j (the held-out folds), averaged over five folds.

In [ ]:
patterns2 = r.normal(size=(C, N)) * 0.2
early = (times >= 80) & (times < 175)
pat_t = np.where(early[None, :, None, None], patterns[None, None], patterns2[None, None])[0]   # bins x C x N
mod = time_course(times)[:, None, None] * pat_t
log_rate2 = np.log(2.0) + mod[:, cats, :].transpose(1, 0, 2) + shared
counts2 = r.poisson(np.exp(log_rate2)).astype(float)

folds = np.array_split(np.arange(n_trials), 5)
G = np.zeros((len(times), len(times)))
for te in folds:
    tr = np.setdiff1d(np.arange(n_trials), te)
    for i in range(len(times)):
        W, b = shrink_lda_fit(counts2[tr, i, :], cats[tr], 0.2)
        for j in range(len(times)):
            G[i, j] += ...
G /= n_trials
print(f"train 100 ms, test 100 ms: {G[list(times).index(100), list(times).index(100)]:.2f}; test 250 ms: {G[list(times).index(100), list(times).index(250)]:.2f}")
print(f"train 250 ms, test 250 ms: {G[list(times).index(250), list(times).index(250)]:.2f}")
plt.imshow(G, origin="lower", extent=[times[0], times[-1] + 25, times[0], times[-1] + 25], vmin=0.25, vmax=1)
plt.xlabel("test time (ms)"); plt.ylabel("training time (ms)"); plt.colorbar(); plt.show()

In [ ]:
import numpy as np
def test_shape():
    "G is a square matrix over time bins"
    assert G.shape == (len(times), len(times))
def test_dynamic():
    "the diagonal is high late, but early-trained decoders fail on late bins"
    j_e, j_l = list(times).index(100), list(times).index(250)
    assert G[j_l, j_l] > 0.6 and G[j_e, j_l] < 0.4

test_shape()
test_dynamic()
print("All checks passed")